# Attendance Trajectories in a parkrun-Style Community Run Series 

**Notebook 01 — Data overview and feasibility check**

---

Mariia Stenina · HSE Faculty of Computer Science · October 2026

In [1]:
import pandas as pd

The data were provided directly by the S95 organisers in October 2026 as a
set of CSV exports, in response to a written request.

In [2]:
activities = pd.read_csv('data/activities.csv')
athletes = pd.read_csv('data/athletes.csv')
clubs = pd.read_csv('data/clubs.csv')
events = pd.read_csv('data/events.csv')
results = pd.read_csv('data/results.csv')
volunteers = pd.read_csv('data/volunteers.csv')

In [3]:
activities = pd.read_csv("data/activities.csv", parse_dates=["date"])
athletes = pd.read_csv("data/athletes.csv", parse_dates=["created_at"])

### `activities` — individual run events (4 378 rows)

| Field | Description | Type |
|---|---|---|
| `id` | Run identifier | int |
| `event_id` | Location, FK to `events.id` | int |
| `date` | Date the run took place | object, 2022-03-12 to 2026-09-26 |
| `published` | Whether results are published | bool, always `True` |

In [4]:
activities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4378 entries, 0 to 4377
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   id         4378 non-null   int64         
 1   event_id   4378 non-null   int64         
 2   date       4378 non-null   datetime64[ns]
 3   published  4378 non-null   bool          
dtypes: bool(1), datetime64[ns](1), int64(2)
memory usage: 107.0 KB


In [5]:
activities.head()

,id,event_id,date,published
0,2,1,2022-06-18,True
1,4,3,2022-07-02,True
2,5,1,2022-07-02,True
3,7,2,2022-07-02,True
4,9,1,2022-06-25,True


In [6]:
activities["date"].agg(["min", "max"])

min   2022-03-12
max   2026-09-26
Name: date, dtype: datetime64[ns]

In [7]:
activities.published.unique()

array([ True])

### `athletes` — registered participants (18 843 rows)

| Field | Description | Type |
|---|---|---|
| `id` | Participant identifier | int |
| `gender` | `male` / `female` | str |
| `club_id` | Running club, FK to `clubs.id` | float, 19% present |
| `event_id` | Self-declared home location, FK to `events.id` | float, 30% present |
| `created_at` | Registration timestamp | datetime, 2022-06-28 to 2026-09-28|
| `fiveverst_code` | Participant's code in the 5 вёрст series | float, 12% present |
| `parkrun_code` | Participant's code in parkrun | float, 11% present |
| `parkzhrun_code` | Participant's code in the ПаркЖРун series | float, 3% present |
| `runpark_code` | Participant's code in the Бег в парке series | float, 4% present |

In [8]:
athletes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18843 entries, 0 to 18842
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              18843 non-null  int64  
 1   gender          18843 non-null  object 
 2   club_id         3518 non-null   float64
 3   event_id        5578 non-null   float64
 4   created_at      18843 non-null  object 
 5   fiveverst_code  2322 non-null   float64
 6   parkrun_code    2093 non-null   float64
 7   parkzhrun_code  639 non-null    float64
 8   runpark_code    812 non-null    float64
dtypes: float64(6), int64(1), object(2)
memory usage: 1.3+ MB


In [9]:
athletes.head()

,id,gender,club_id,event_id,created_at,fiveverst_code,parkrun_code,parkzhrun_code,runpark_code
0,1,male,1.0,1.0,2022-06-30 21:00:00,NaN,875743.0,NaN,NaN
1,3,male,22.0,1.0,2022-06-28 08:38:49.291872,NaN,875431.0,NaN,NaN
2,6,male,6.0,3.0,2022-06-28 08:38:49.318625,NaN,876288.0,NaN,NaN
3,7,male,NaN,NaN,2022-06-28 08:38:49.327216,NaN,925293.0,NaN,NaN
4,10,male,NaN,NaN,2022-06-28 08:38:49.362636,790059705.0,1913947.0,NaN,NaN


In [10]:
athletes["created_at"].agg(["min", "max"])

min    2022-06-28 08:38:49.291872
max    2026-09-28 11:03:56.589234
Name: created_at, dtype: object

In [11]:
cols = ["club_id", "event_id", "fiveverst_code", "parkrun_code",
        "parkzhrun_code", "runpark_code"]
athletes[cols].notna().mean().round(3).rename("share_present").to_frame()

,share_present
club_id,0.187
event_id,0.296
fiveverst_code,0.123
parkrun_code,0.111
parkzhrun_code,0.034
runpark_code,0.043


### `clubs` — running clubs (264 rows)

| Field | Description | Type |
|---|---|---|
| `id` | Club identifier | int |
| `name` | Club name | str |

In [12]:
clubs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264 entries, 0 to 263
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      264 non-null    int64 
 1   name    264 non-null    object
dtypes: int64(1), object(1)
memory usage: 4.3+ KB


In [13]:
clubs.head()

,id,name
0,1,Wake&Run
1,3,Академия марафона
2,4,Mikkeller Running Club Moscow
3,5,ЛК Измайлово
4,6,IzMyLong


### `events` — locations (38 rows)

| Field | Description | Type |
|---|---|---|
| `id` | Location identifier | int |
| `code_name` | Latin slug used in site URLs | str |
| `name` | Location name | str |
| `town` | Town or city | str |
| `latitude` | Latitude, decimal degrees | float, 2 nulls |
| `longitude` | Longitude, decimal degrees | float, 2 nulls |
| `active` | Whether the location is currently running | bool, always `True` |

Two rows are not physical parks but aggregate records for guest results
(`С95 и друзья`, Russia; `S95 & Friends`, Belarus) — hence the missing
coordinates.

In [14]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         38 non-null     int64  
 1   code_name  38 non-null     object 
 2   name       38 non-null     object 
 3   town       38 non-null     object 
 4   latitude   36 non-null     float64
 5   longitude  36 non-null     float64
 6   active     38 non-null     bool   
dtypes: bool(1), float64(2), int64(1), object(3)
memory usage: 1.9+ KB


In [15]:
events.head()

,id,code_name,name,town,latitude,longitude,active
0,1,kuzminki,Кузьминки,Москва,55.691925,37.772094,True
1,2,olimpiyskaya_derevnya,Олимпийская Деревня,Москва,55.680814,37.480172,True
2,3,izmailovo,Измайлово,Москва,55.788149,37.794292,True
3,4,s,С95 и друзья,Россия,NaN,NaN,True
4,5,tsaritsyno,Царицыно,Москва,55.622017,37.679497,True


In [16]:
events[events["latitude"].isna()]

,id,code_name,name,town,latitude,longitude,active
3,4,s,С95 и друзья,Россия,NaN,NaN,True
29,31,bs,S95 & Friends,Беларусь,NaN,NaN,True


### `results` — attendance records (146 483 rows)

| Field | Description | Type |
|---|---|---|
| `activity_id` | Run attended, FK to `activities.id` | int |
| `athlete_id` | Participant, FK to `athletes.id` | float, 5.4% null |
| `total_time` | Finish time in seconds (median 28:09) | int |
| `position` | Finishing place within that run (1–845) | int |
| `first_run` | Participant's first ever S95 run | bool |
| `personal_best` | Result was a personal best | bool |

In [17]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146483 entries, 0 to 146482
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   activity_id    146483 non-null  int64  
 1   athlete_id     138534 non-null  float64
 2   total_time     146483 non-null  int64  
 3   position       146483 non-null  int64  
 4   first_run      146483 non-null  bool   
 5   personal_best  146483 non-null  bool   
dtypes: bool(2), float64(1), int64(3)
memory usage: 4.7 MB


In [18]:
results.head()

,activity_id,athlete_id,total_time,position,first_run,personal_best
0,2,3365.0,980,1,False,True
1,2,856.0,981,2,False,True
2,2,908.0,1045,3,False,True
3,2,2625.0,1056,4,False,False
4,2,853.0,1069,5,False,True


### `volunteers` — volunteering records (47 729 rows)

| Field | Description | Type |
|---|---|---|
| `activity_id` | Run volunteered at, FK to `activities.id` | int |
| `athlete_id` | Participant, FK to `athletes.id` | int, no nulls |
| `role` | Role performed, 25 distinct values | str |

In [19]:
volunteers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47729 entries, 0 to 47728
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   activity_id  47729 non-null  int64 
 1   athlete_id   47729 non-null  int64 
 2   role         47729 non-null  object
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


In [20]:
volunteers.head()

,activity_id,athlete_id,role
0,2,155,scanner
1,2,194,scanner
2,2,223,finish_maker
3,2,679,timer
4,2,955,results_handler


Разбивка по ролям волонтеров:

In [21]:
volunteers['role'].value_counts().to_frame('n')

,n
role,
timer,5053
scanner,4542
event_closer,4284
director,3877
tokens,3669
photograph,3477
pacemaker,3284
marshal,2835
marking_maker,2817


Сколько людей обслуживает 1 забег?

In [22]:
volunteers.groupby('activity_id').size().describe()

count    4377.000000
mean       10.904501
std         8.365318
min         1.000000
25%         6.000000
50%         9.000000
75%        14.000000
max       116.000000
dtype: float64

Сколько раз волонтерит 1 человек?

In [23]:
volunteers.groupby('athlete_id').size().describe()

count    4302.000000
mean       11.094607
std        22.897985
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       224.000000
dtype: float64